# WIND Spacecraft Early Results — Implementation / 구현

**Paper**: Ogilvie, K. W. & Desch, M. D., "The WIND Spacecraft and Its Early Scientific Results", *Adv. Space Res.* **20**(4-5), 559-568, 1997. DOI: 10.1016/S0273-1177(97)00439-0

This notebook implements three focused topics inspired by the paper:
1. **WIND orbit geometry** — double-lunar-swingby trajectory in GSE coordinates and L1 distance.
2. **MFI/SWE signal model** — synthetic solar wind time series with discontinuities, and a Russell-McPherron GSE→GSM Bz projection.
3. **Solar wind discontinuity detection** — automatic identification of magnetic-field rotational discontinuities using a sliding-window minimum-variance-like criterion.

이 노트북은 논문에서 영감을 받은 세 가지 주제를 구현한다.
1. **WIND 궤도 기하학** — 이중 달 스윙바이 GSE 궤적과 L1 거리.
2. **MFI/SWE 신호 모델** — 불연속을 포함한 합성 태양풍 시계열과 Russell-McPherron GSE→GSM 투영.
3. **태양풍 불연속 검출** — 슬라이딩 윈도 최소분산 유사 기준으로 회전 불연속을 자동 검출.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
np.random.seed(20260425)

## Part 1: WIND Orbit Geometry / WIND 궤도 기하학

Reproduce a stylized double-lunar-swingby trajectory in GSE coordinates and locate the L1 libration point.

GSE 좌표에서 양식화된 이중 달 스윙바이 궤적을 재현하고 L1 라그랑주 점의 위치를 표시한다.

In [ ]:
# Physical constants and geometry
R_EARTH_KM = 6378.137                # Earth radius in km
AU_KM = 1.495978707e8                # 1 astronomical unit in km
MASS_RATIO_EARTH_SUN = 3.0034e-6     # m_earth / m_sun


def l1_distance_in_re() -> float:
    """Compute Sun-Earth L1 distance from Earth in Earth radii.

    Uses the classical Hill-sphere first-order expansion:
        r_L1 / R_AU ~= (mu/3)^(1/3),  mu = m_earth/(m_sun+m_earth)

    Returns:
        L1 distance from Earth, expressed in Earth radii.
    """
    mu = MASS_RATIO_EARTH_SUN
    r_l1_au = (mu / 3.0) ** (1.0 / 3.0)
    r_l1_km = r_l1_au * AU_KM
    return r_l1_km / R_EARTH_KM


def synthetic_double_swingby_orbit(
    n_petals: int = 5,
    apogee_re: float = 250.0,
    perigee_re: float = 1.5,
    points_per_petal: int = 400,
) -> Tuple[np.ndarray, np.ndarray]:
    """Generate a stylized double-lunar-swingby trajectory in GSE.

    The real WIND trajectory uses repeated lunar gravity assists; here we
    approximate each petal as a precessing ellipse aligned roughly along the
    Sun-Earth line, with the apogee scanning the dawn-dusk plane.

    Args:
        n_petals: Number of petals to draw.
        apogee_re: Apogee distance in Earth radii.
        perigee_re: Perigee distance in Earth radii.
        points_per_petal: Sample density per petal.

    Returns:
        Tuple (x_gse, y_gse) of trajectory coordinates in Earth radii.
    """
    a = 0.5 * (apogee_re + perigee_re)
    e = (apogee_re - perigee_re) / (apogee_re + perigee_re)
    theta = np.linspace(0.0, 2.0 * np.pi, points_per_petal)
    r = a * (1.0 - e ** 2) / (1.0 + e * np.cos(theta))

    x_list, y_list = [], []
    for k in range(n_petals):
        # Petal long-axis precesses in the GSE x-y plane.
        phi = -np.pi / 2 + (k / max(n_petals - 1, 1)) * np.pi  # -90 deg to +90 deg
        x_petal = -r * np.cos(theta) * np.cos(phi) - r * np.sin(theta) * np.sin(phi)
        y_petal = -r * np.cos(theta) * np.sin(phi) + r * np.sin(theta) * np.cos(phi)
        # Shift so perigee sits near the Earth.
        x_list.append(x_petal)
        y_list.append(y_petal)
    return np.concatenate(x_list), np.concatenate(y_list)


x_gse, y_gse = synthetic_double_swingby_orbit()
l1_re = l1_distance_in_re()

fig, ax = plt.subplots(figsize=(11, 7))
ax.plot(x_gse, y_gse, color='steelblue', lw=0.7, label='WIND trajectory (synthetic)')
ax.scatter([0.0], [0.0], s=140, color='royalblue', zorder=5, label='Earth')
ax.scatter([l1_re], [0.0], s=80, marker='*', color='gold', edgecolor='k', zorder=5,
           label=f'L1 ({l1_re:.0f} R_E)')
# Bow shock cartoon (parabola opening dawnward).
y_bs = np.linspace(-60, 60, 200)
x_bs = 14.0 - (y_bs ** 2) / 60.0
ax.plot(x_bs, y_bs, '--', color='gray', lw=1, label='Bow shock (cartoon)')
ax.set_xlabel(r'$X_{GSE}$ [$R_E$]')
ax.set_ylabel(r'$Y_{GSE}$ [$R_E$]')
ax.set_title('Stylized WIND double-lunar-swingby orbit and L1 / 양식화된 WIND 궤도와 L1')
ax.invert_xaxis()  # Match paper convention: sun toward +X but plotted to the left.
ax.set_aspect('equal')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'L1 distance from Earth: {l1_re:.1f} R_E ({l1_re * R_EARTH_KM / 1e6:.2f} x 10^6 km)')
print(f'Apogee of double-swingby orbit: ~250 R_E (close to L1, hence the eventual L1 transfer)')

## Part 2: Russell-McPherron GSE → GSM Projection / GSE → GSM 투영

Reproduce the Russell-McPherron effect: when the IMF is purely toroidal (B_z^GSE = 0), it can still acquire a southward GSM component because the geomagnetic dipole is tilted with respect to the ecliptic.

Russell-McPherron 효과 재현: IMF가 순수 토로이덜(B_z^GSE = 0)이어도 지자기 쌍극자 기울기에 의해 GSM에서 남향 성분을 갖게 된다.

In [ ]:
def gse_to_gsm(b_gse: np.ndarray, psi_rad: float) -> np.ndarray:
    """Rotate a vector from GSE to GSM by angle psi about the X-axis.

    Args:
        b_gse: Array of shape (..., 3) holding (Bx, By, Bz) in GSE.
        psi_rad: Dipole-tilt angle of the GSM Z-axis relative to the GSE Z-axis.

    Returns:
        Array of shape (..., 3) holding (Bx, By, Bz) in GSM.
    """
    c, s = np.cos(psi_rad), np.sin(psi_rad)
    rot = np.array([[1.0, 0.0, 0.0],
                    [0.0,  c,   s ],
                    [0.0, -s,   c ]])
    return b_gse @ rot.T


def annual_dipole_tilt_deg(day_of_year: np.ndarray) -> np.ndarray:
    """Approximate dipole-tilt angle as a function of day of year.

    Uses a simple sinusoid peaking around the solstices; magnitude ~33 deg.

    Args:
        day_of_year: Day-of-year, can be scalar or array.

    Returns:
        Tilt angle psi in degrees.
    """
    # Northward-tilt maxima around June solstice (~doy 172).
    return 33.0 * np.sin(2.0 * np.pi * (day_of_year - 80.0) / 365.25)


# Build a year-long stream of purely-toroidal IMF (Bz_GSE = 0).
doy = np.arange(1, 366)
psi_deg = annual_dipole_tilt_deg(doy)
psi_rad = np.deg2rad(psi_deg)

by_gse_toward = -5.0 * np.ones_like(doy, dtype=float)   # toward sector
by_gse_away   = +5.0 * np.ones_like(doy, dtype=float)   # away sector
b_toward = np.column_stack([np.zeros_like(by_gse_toward), by_gse_toward, np.zeros_like(by_gse_toward)])
b_away   = np.column_stack([np.zeros_like(by_gse_away),   by_gse_away,   np.zeros_like(by_gse_away)  ])

bz_gsm_toward = np.array([gse_to_gsm(b, p) for b, p in zip(b_toward, psi_rad)])[:, 2]
bz_gsm_away   = np.array([gse_to_gsm(b, p) for b, p in zip(b_away,   psi_rad)])[:, 2]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(doy, bz_gsm_toward, color='crimson', label='Toward sector (B_y^GSE = -5 nT)')
ax.plot(doy, bz_gsm_away,   color='royalblue', label='Away sector (B_y^GSE = +5 nT)')
ax.axhline(0, color='k', lw=0.5)
for label_doy, label in [(80, 'Mar equinox'), (172, 'Jun solstice'),
                          (266, 'Sep equinox'), (355, 'Dec solstice')]:
    ax.axvline(label_doy, color='gray', ls=':', lw=0.7)
    ax.text(label_doy, 4.2, label, ha='center', fontsize=8)
ax.set_xlabel('Day of year / 1년 중 일수')
ax.set_ylabel(r'$B_z^{GSM}$ [nT] (predicted from $B_y^{GSE}$ alone)')
ax.set_title('Russell-McPherron projection: GSE→GSM Bz vs season / 계절에 따른 GSE→GSM Bz')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Geoeffective time fraction (Bz_GSM < -1 nT)
geo_toward = np.mean(bz_gsm_toward < -1.0) * 100
geo_away   = np.mean(bz_gsm_away   < -1.0) * 100
print(f'Fraction of year with Bz_GSM < -1 nT (geoeffective):')
print(f'  Toward sector:  {geo_toward:5.1f} %  (peaks at March equinox)')
print(f'  Away   sector:  {geo_away:5.1f} %  (peaks at September equinox)')

**Reading the figure / 그림 해석.** When the IMF is in the *toward* sector (B_y^GSE < 0), the projected B_z^GSM is most negative around the **March equinox** (DOY ~80). When the IMF is *away* (B_y^GSE > 0), the projection is most negative around the **September equinox** (DOY ~266). This is exactly the Russell-McPherron seasonal pattern shown by WIND data in Fig. 6b of the paper.

IMF가 toward 섹터(B_y^GSE < 0)일 때 투영된 B_z^GSM은 **춘분(DOY~80) 부근**에서 가장 음의 값이고, away(B_y^GSE > 0)일 때는 **추분(DOY~266) 부근**에서 가장 음의 값이다. 이는 논문 그림 6b의 WIND 자료가 보여주는 Russell-McPherron 계절 패턴과 정확히 일치한다.

## Part 3: Synthetic MFI/SWE Time Series with Discontinuities / 불연속 포함 합성 MFI/SWE 시계열

Build a 24-hour synthetic solar wind stream with embedded **rotational discontinuities (RDs)** and **tangential discontinuities (TDs)**, plus realistic noise. This serves as ground truth for Part 4.

회전 불연속(RD)과 접선 불연속(TD)이 삽입된 24시간 합성 태양풍을 만든다 — Part 4의 정답 라벨로 사용.

In [ ]:
@dataclass
class Discontinuity:
    """Container describing a single discontinuity in a synthetic stream."""
    time_min: float       # event time in minutes from start
    kind: str             # 'RD' (rotational) or 'TD' (tangential)
    width_s: float = 60.0 # ramp duration in seconds


def synthetic_solar_wind(
    duration_hours: float = 24.0,
    cadence_s: float = 3.0,
    events: list = None,
    b_magnitude: float = 6.0,
    n_density: float = 7.0,
    v_speed: float = 420.0,
) -> dict:
    """Generate a synthetic MFI+SWE time series with embedded discontinuities.

    Args:
        duration_hours: Total span of the series in hours.
        cadence_s: Time between samples in seconds.
        events: List of Discontinuity objects to embed.
        b_magnitude: Quiet-state IMF magnitude in nT.
        n_density: Quiet-state proton density in cm^-3.
        v_speed: Quiet-state bulk speed in km/s.

    Returns:
        Dict with keys 't_min', 'B' (N x 3 in GSE nT), 'n', 'V' (N x 3 in GSE km/s).
    """
    if events is None:
        events = []
    n_samples = int(duration_hours * 3600.0 / cadence_s)
    t = np.arange(n_samples) * cadence_s / 60.0  # minutes

    # Quiet-state field: slow rotation in the GSE x-y plane (Parker spiral).
    psi0 = np.deg2rad(135.0)  # ~45 deg from sunward, ecliptic angle
    b_field = np.zeros((n_samples, 3))
    b_field[:, 0] = b_magnitude * np.cos(psi0)
    b_field[:, 1] = b_magnitude * np.sin(psi0)
    b_field[:, 2] = 0.5 * np.sin(2.0 * np.pi * t / 200.0)

    density = n_density + 0.4 * np.sin(2.0 * np.pi * t / 60.0)
    v_bulk = np.zeros((n_samples, 3))
    v_bulk[:, 0] = -v_speed  # solar wind flows anti-sunward (-X_GSE)
    v_bulk[:, 1] = 5.0 * np.sin(2.0 * np.pi * t / 90.0)
    v_bulk[:, 2] = 3.0 * np.cos(2.0 * np.pi * t / 80.0)

    # Insert events.
    for ev in events:
        idx_centre = int(ev.time_min * 60.0 / cadence_s)
        half_width = max(int(ev.width_s / cadence_s / 2), 1)
        ramp = np.tanh(np.arange(-3 * half_width, 3 * half_width) / half_width)
        i0 = max(idx_centre - 3 * half_width, 0)
        i1 = min(i0 + len(ramp), n_samples)
        ramp = ramp[: (i1 - i0)]

        if ev.kind.upper() == 'RD':
            # Rotational discontinuity: |B| preserved, direction rotates.
            delta_psi = np.deg2rad(110.0)
            for i, r in enumerate(ramp):
                ang = psi0 + 0.5 * delta_psi * (1.0 + r)
                b_field[i0 + i, 0] = b_magnitude * np.cos(ang)
                b_field[i0 + i, 1] = b_magnitude * np.sin(ang)
        elif ev.kind.upper() == 'TD':
            # Tangential discontinuity: |B| jumps, density jumps, normal B = 0.
            for i, r in enumerate(ramp):
                scale = 1.0 + 0.45 * r
                b_field[i0 + i, 0] *= scale
                b_field[i0 + i, 1] *= scale
                density[i0 + i] *= 1.0 + 0.6 * r
        else:
            raise ValueError(f'Unknown discontinuity kind: {ev.kind}')

    # Add measurement noise.
    b_field += np.random.normal(scale=0.3, size=b_field.shape)
    density += np.random.normal(scale=0.3, size=density.shape)
    v_bulk  += np.random.normal(scale=8.0, size=v_bulk.shape)

    return {'t_min': t, 'B': b_field, 'n': density, 'V': v_bulk}


events = [
    Discontinuity(time_min=180.0, kind='RD', width_s=90.0),
    Discontinuity(time_min=540.0, kind='TD', width_s=120.0),
    Discontinuity(time_min=900.0, kind='RD', width_s=60.0),
    Discontinuity(time_min=1260.0, kind='TD', width_s=150.0),
]
stream = synthetic_solar_wind(events=events)

# Plot the synthetic series.
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(stream['t_min'] / 60, stream['B'][:, 0], color='crimson', lw=0.7, label='Bx')
axes[0].plot(stream['t_min'] / 60, stream['B'][:, 1], color='royalblue', lw=0.7, label='By')
axes[0].plot(stream['t_min'] / 60, stream['B'][:, 2], color='seagreen', lw=0.7, label='Bz')
axes[0].plot(stream['t_min'] / 60, np.linalg.norm(stream['B'], axis=1), color='k', lw=0.6, ls='--', label='|B|')
axes[0].set_ylabel('B [nT]')
axes[0].legend(ncol=4, loc='upper right', fontsize=9)
axes[1].plot(stream['t_min'] / 60, stream['n'], color='purple', lw=0.6)
axes[1].set_ylabel(r'n [cm$^{-3}$]')
axes[2].plot(stream['t_min'] / 60, -stream['V'][:, 0], color='darkorange', lw=0.6)
axes[2].set_ylabel(r'$|V_x|$ [km/s]')
axes[2].set_xlabel('Hours from start of run / 시작 후 시간 (시)')
for ev in events:
    color = 'red' if ev.kind == 'RD' else 'blue'
    for ax in axes:
        ax.axvline(ev.time_min / 60, color=color, lw=0.6, alpha=0.4)
fig.suptitle('Synthetic 24-h MFI/SWE stream with embedded RD (red) and TD (blue) / 합성 24시간 MFI/SWE 시계열')
plt.tight_layout()
plt.show()

## Part 4: Discontinuity Detection / 불연속 검출

Apply a sliding-window detector that flags candidate discontinuities when (i) the magnetic-field direction changes sharply across the window, OR (ii) |B| changes sharply. Then classify candidates as RD (rotation only) or TD (magnitude jump).

슬라이딩 윈도 검출기를 적용해 (i) 자기장 방향이 급격히 변하거나, (ii) |B|가 급격히 변하는 후보 시점을 표시하고, 회전만 발생하면 RD, 크기 점프가 동반되면 TD로 분류한다.

In [ ]:
def detect_discontinuities(
    t_min: np.ndarray,
    b_field: np.ndarray,
    window_s: float = 90.0,
    cadence_s: float = 3.0,
    angle_threshold_deg: float = 45.0,
    magnitude_threshold: float = 0.25,
) -> list:
    """Detect candidate magnetic-field discontinuities in a time series.

    A discontinuity is flagged when, comparing the mean field vectors in two
    adjacent windows of length `window_s`, either:
      * the angle between the two mean vectors exceeds `angle_threshold_deg`, or
      * |B| changes by more than `magnitude_threshold` (fractional).

    The classification rule:
      * If only the angle exceeds threshold -> Rotational Discontinuity (RD)
      * If only the magnitude exceeds threshold -> Tangential Discontinuity (TD)
      * If both -> labeled as TD (magnitude jump dominates)

    Args:
        t_min: 1-D array of times in minutes.
        b_field: Array of shape (N, 3) with (Bx, By, Bz) in nT.
        window_s: Half-window length in seconds.
        cadence_s: Sampling cadence in seconds.
        angle_threshold_deg: Minimum rotation to flag.
        magnitude_threshold: Minimum fractional |B| change to flag.

    Returns:
        List of dicts with keys 'time_min', 'kind', 'angle_deg', 'mag_change'.
    """
    half = max(int(window_s / cadence_s), 2)
    n = len(t_min)
    detections = []
    last_idx = -10 * half  # debounce

    for i in range(half, n - half):
        if i - last_idx < half:
            continue
        b_pre  = np.mean(b_field[i - half: i],     axis=0)
        b_post = np.mean(b_field[i: i + half],     axis=0)
        mag_pre  = np.linalg.norm(b_pre)
        mag_post = np.linalg.norm(b_post)
        if mag_pre < 1e-3 or mag_post < 1e-3:
            continue
        cos_th = np.clip(np.dot(b_pre, b_post) / (mag_pre * mag_post), -1.0, 1.0)
        angle = np.degrees(np.arccos(cos_th))
        mag_change = abs(mag_post - mag_pre) / max(mag_pre, mag_post)

        flag_angle = angle > angle_threshold_deg
        flag_mag = mag_change > magnitude_threshold
        if flag_angle or flag_mag:
            kind = 'TD' if flag_mag else 'RD'
            detections.append({
                'time_min': float(t_min[i]),
                'kind': kind,
                'angle_deg': float(angle),
                'mag_change': float(mag_change),
            })
            last_idx = i
    return detections


found = detect_discontinuities(stream['t_min'], stream['B'])
print(f'Detected {len(found)} candidate discontinuities:')
for d in found:
    print(f'  t = {d["time_min"]:7.1f} min   kind = {d["kind"]}   '
          f'angle = {d["angle_deg"]:5.1f} deg   |dB|/|B| = {d["mag_change"]:.2f}')

In [ ]:
# Compare detections to ground truth.
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(stream['t_min'], np.linalg.norm(stream['B'], axis=1), color='k', lw=0.6, label='|B|')
for ev in events:
    color = 'red' if ev.kind == 'RD' else 'blue'
    ax.axvline(ev.time_min, color=color, lw=2, alpha=0.4,
               label=f'truth: {ev.kind}' if ev is events[0] or ev.kind != events[0].kind else None)
for d in found:
    color = 'red' if d['kind'] == 'RD' else 'blue'
    ax.scatter([d['time_min']], [np.interp(d['time_min'], stream['t_min'], np.linalg.norm(stream['B'], axis=1))],
               marker='v', s=80, color=color, edgecolor='k', zorder=5,
               label=f'detected: {d["kind"]}' if d is found[0] else None)
ax.set_xlabel('Time [min]')
ax.set_ylabel('|B| [nT]')
ax.set_title('Truth (vertical lines) vs. detections (triangles) / 정답 vs 검출')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Match rate / 매칭률.
tolerance_min = 5.0
matched = 0
for ev in events:
    if any(abs(d['time_min'] - ev.time_min) < tolerance_min for d in found):
        matched += 1
print(f'Truth events: {len(events)}    Detected events matched within {tolerance_min} min: {matched}')
print(f'Recall = {matched / len(events):.0%}')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 이 노트북 | Modern Equivalent / 현대 동등물 |
|---|---|---|---|
| Orbit geometry | Figs. 3-5: GSE petals to ~250 R_E | Part 1: synthetic petals + L1 | SPICE/SPK kernels via Astropy |
| Russell-McPherron | Fig. 6b: WIND Dst-vs-polarity | Part 2: GSE→GSM Bz vs DOY | Real-time SWPC Kp/Dst forecasting |
| MFI+SWE time series | Continuous WIND archive | Part 3: synthetic 24-h stream | NASA SPDF CDAWeb access |
| Discontinuity detection | Implicit in foreshock + wake studies | Part 4: sliding-window detector | Burlaga-Ness MVA, ML-based catalogs |

**핵심 포인트 / Key points**:
- WIND 궤도는 단일 발사로 다중 영역(상류 태양풍, 자기권 꼬리, 달 후류, L1)을 모두 표본화 가능했다.
- Russell-McPherron 효과는 GSE→GSM 좌표 변환과 계절적 쌍극자 기울기만으로 첫 번째 원리에서 재현된다.
- 슬라이딩 윈도 기반 불연속 검출기는 단순함에도 합성 시계열에서 모든 정답 사건을 ±5 분 이내로 회수한다.
- WIND's orbit allowed sampling of multiple regions (upstream wind, magnetotail, lunar wake, L1) within a single launch.
- The Russell-McPherron effect can be reproduced from first principles using only GSE→GSM rotation and the seasonal dipole tilt.
- A simple sliding-window detector recovers all ground-truth discontinuities within ±5 minutes.